<a href="https://colab.research.google.com/github/prjhseo/study_RS/blob/main/Factorization_Machines_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FastFM
- 필요하면 관련 논문 찾아보기



In [ ]:
!pip install fastfm

In [ ]:
!wget https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_comics_graphic.json.gz

In [13]:
import gzip
import json

user_ids={}
book_ids={}

data=[]

with gzip.open('goodreads_reviews_comics_graphic.json.gz') as f:
    for line in f:
        record = json.loads(line) # json string -> dictionary

        uid=record['user_id']
        bid=record['book_id']
        rating=record['rating']
        n_votes=record['n_votes']
        n_comments=record['n_comments']

        if uid not in user_ids:
            user_ids[uid]=len(user_ids)
        if bid not in book_ids:
            book_ids[bid]=len(book_ids)

        data.append((user_ids[uid],book_ids[bid],n_votes,n_comments,rating))

n_users=len(user_ids)
n_books=len(book_ids)



# 데이터 준비

In [14]:
from scipy.sparse import csr_matrix
import numpy as np
import random

random.shuffle(data) # 학습이 치우치지 않도록 셔플

#CSR을 만들기 위한 과
rows=[]
cols=[]
values=[]

n_votes_idx=n_users+n_books # 따봉 몇번
n_comments_idx=n_users+n_books+1 # 답글 수

for rowid, (uid,bid,n_votes,n_comments,rating) in enumerate(data):
    rows.extend([rowid,rowid,rowid,rowid])
    cols.extend([uid,n_users+bid,n_votes_idx,n_comments_idx])
    values.extend([1,1,n_votes,n_comments])

X=csr_matrix((values,(rows,cols)))
y=np.array([rating for _,_,_,_,rating in data],dtype=np.float64)

X_train,y_train=X[:40000],y[:40000]
X_test,y_test=X[40000:],y[40000:]

In [15]:
from fastFM import als

fm=als.FMRegression(n_iter=1000,init_stdev=0.1,rank=5, l2_reg_w=0.1,l2_reg_V=0.5)
fm.fit(X_train,y_train)

FMRegression(l2_reg_V=0.5, n_iter=1000, rank=5)

- n_iter: 학습 반복 수
- init_stdev : 파라미터 초기화 변수
- rank : latent vector 차원
- I2_reg_w : weight 파라미터 규제 가중치
- I2_reg_V : latent vector 파라미터 규제 가중치

In [16]:
y_pred=fm.predict(X_test)
print(y_pred[:10])
print(y_test[:10])
print("RMSE: ",((y_test-y_pred)**2).mean()**0.5)

[2.76783427 3.95846779 2.87863237 4.18840242 4.15190542 3.39122269
 3.0998648  3.12513177 4.28637609 3.97329864]
[3. 4. 4. 4. 5. 3. 4. 2. 3. 3.]
RMSE:  14.105651509997315
